In [4]:
import catboost as cb

3. Как работает обработка категориальных признаков:
Target Encoding (Ordered Target Statistics)

# Вместо one-hot encoding CatBoost использует:
# 1. Для регрессии: среднее целевой переменной по категории
# 2. Для классификации: вероятность класса по категории
# 3. Добавляет шум для предотвращения переобучения

# Пример: категория "город"
# Москва: средний таргет = 0.65
# СПб: средний таргет = 0.72
# Казань: средний таргет = 0.58


# Обычный градиентный бустинг:
# 1. Считает градиенты на всей выборке
# 2. Строит дерево
# 3. Проблема: переобучение (target leakage)

# Ordered Boosting в CatBoost:
# 1. Для каждого объекта использует только предыдущие объекты
# 2. Исключает target leakage
# 3. Защита от переобучения

In [10]:
import catboost as cb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Создание данных с категориальными признаками
data = pd.DataFrame({
    'age': [25, 30, 35, 40, 45, 50],
    'city': ['Moscow', 'SPb', 'Moscow', 'Kazan', 'SPb', 'Moscow'],  # Категориальный
    'income': [50000, 60000, 55000, 70000, 65000, 80000],
    'education': ['high', 'medium', 'high', 'low', 'medium', 'high'],  # Категориальный
    'target': [1, 0, 1, 0, 1, 0]
})

# Разделение на признаки и целевую переменную
X = data.drop('target', axis=1)
y = data['target']

# Указание категориальных признаков
cat_features = ['city', 'education']

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Создание модели
model = cb.CatBoostClassifier(
    iterations=1000,  # Количество деревьев
    learning_rate=0.1,
    depth=6,  # Глубина деревьев
    loss_function='Logloss',  # Функция потерь
    eval_metric='AUC',  # Метрика для валидации
    cat_features=cat_features,  # Категориальные признаки
    random_seed=42,
    verbose=100,  # Вывод каждые 100 итераций
    early_stopping_rounds=50  # Ранняя остановка
)

# Обучение с валидацией
model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    plot=True  # Построение графиков обучения
)

# Предсказание
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Оценка
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	test: 0.5000000	best: 0.5000000 (0)	total: 57.1ms	remaining: 57.1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1
bestIteration = 7

Shrink model to first 8 iterations.
Accuracy: 0.5000
ROC-AUC: 1.0000


In [5]:
model = cb.CatBoostClassifier(
    # Основные параметры
    iterations=1000,  # Количество деревьев
    learning_rate=0.03,  # Скорость обучения (лучше 0.03-0.1)
    depth=6,  # Глубина деревьев (6-10)
    
    # Регуляризация
    l2_leaf_reg=3,  # L2 регуляризация
    random_strength=1,  # Сила случайности
    bagging_temperature=1,  # Температура бэггинга
    
    # Работа с категориями
    one_hot_max_size=2,  # One-hot для категорий с <= 2 уникальных значений
    has_time=False,  # Есть ли временной порядок
    
    # Контроль переобучения
    early_stopping_rounds=50,
    use_best_model=True,
    
    # Производительность
    thread_count=-1,  # Все ядра
    task_type='CPU',  # 'CPU' или 'GPU'
    
    # Другие
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)


In [ ]:
# Gridsearch with CatBoost

from sklearn.model_selection import GridSearchCV

# Определение параметров для поиска
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7],
    'iterations': [500, 1000]}

# Создание модели
model = cb.CatBoostClassifier(
    cat_features=cat_features,
    random_seed=42,
    verbose=0)

# Grid Search
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1)

grid_search.fit(X_train, y_train)
print(f"Лучшие параметры: {grid_search.best_params_}")

In [ ]:
# Практические советы:

# 1. Всегда указывайте cat_features
# 2. Используйте early_stopping_rounds
# 3. Начинайте с learning_rate=0.03-0.05
# 4. Для GPU: depth=8-10, для CPU: depth=6-8
# 5. Используйте Pool для больших данных
# 6. Для текстовых категорий используйте TextFeatures

# Пример с текстовыми признаками
model = cb.CatBoostClassifier(
    text_features=['description', 'title'],  # Текстовые колонки
    tokenizers=[{'tokenizer_id': 'Sense', 'delimiter': ' ', 'lowercasing': 'true'}],
    dictionaries=[{'dictionary_id': 'Word', 'gram_order': '1'}],
    feature_calcers=['BoW', 'NaiveBayes'],
    iterations=1000)